# TDSal: Task-Driven Visual Saliency Prediction

Supplementary code for "TDSal: A Task-Based Top-Down Saliency Prediction Model" (CGI 2026).

**Expected dataset layout at `DATA_PATH`:**
```
DATA_PATH/
  stimuli/       # input images (.jpg or .png)
  task1/fdm/     # free view fixation density maps
  task2/fdm/     # count people
  task3/fdm/     # detect the emotion
  task4/fdm/     # identify the action
```

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

%pip install --quiet einops ultralytics 'sentence-transformers<4.0.0' matplotlib

In [ ]:
import os, glob, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from einops import rearrange
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from ultralytics import YOLO
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DATA_PATH = '/content/drive/MyDrive/TDSP/Task-based-eye-fixation-dataset_1024x768'
CKPT_PATH = '/content/drive/MyDrive/TDSP/models/tdsp_50epoch.pth'

In [ ]:
class PairedRandomHorizontalFlip:
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img, sal):
        if random.random() < self.p:
            img, sal = TF.hflip(img), TF.hflip(sal)
        return img, sal

class PairedRandomRotation:
    def __init__(self, degrees=10): self.degrees = degrees
    def __call__(self, img, sal):
        angle = random.uniform(-self.degrees, self.degrees)
        return TF.rotate(img, angle), TF.rotate(sal, angle)

In [ ]:
task_mapping = {
    'task1': 'free view',
    'task2': 'count people',
    'task3': 'detect the emotion',
    'task4': 'identify the action',
}

class TaskSaliencyDataset(Dataset):
    def __init__(self, data_root, task_mapping, transform=None,
                 saliency_transform=None, paired_transforms=None):
        self.data_root = data_root
        self.task_mapping = task_mapping
        self.transform = transform
        self.saliency_transform = saliency_transform
        self.paired_transforms = paired_transforms or []
        self.samples = []
        for task in task_mapping:
            fdm_folder = os.path.join(data_root, task, 'fdm')
            for fdm_file in glob.glob(os.path.join(fdm_folder, '*.png')):
                base = os.path.splitext(os.path.basename(fdm_file))[0]
                for ext in ('.jpg', '.png'):
                    sp = os.path.join(data_root, 'stimuli', base + ext)
                    if os.path.exists(sp):
                        self.samples.append((sp, fdm_file, task))
                        break

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        stim_path, fdm_path, task = self.samples[idx]
        img = Image.open(stim_path).convert('RGB')
        fdm = Image.open(fdm_path).convert('L')
        if self.transform:          img = self.transform(img)
        if self.saliency_transform: fdm = self.saliency_transform(fdm)
        else:                       fdm = T.ToTensor()(fdm)
        for t in self.paired_transforms:
            img, fdm = t(img, fdm)
        return {'stimuli': img, 'fdm': fdm, 'task': task,
                'task_description': self.task_mapping[task]}

img_tf = T.Compose([T.Resize((384, 384)), T.ToTensor()])
sal_tf = T.Compose([T.Resize((384, 384)), T.ToTensor()])

dataset = TaskSaliencyDataset(
    DATA_PATH, task_mapping,
    transform=img_tf, saliency_transform=sal_tf,
    paired_transforms=[PairedRandomHorizontalFlip(), PairedRandomRotation()],
)
print(f'Total samples: {len(dataset)}')

In [ ]:
total_size = len(dataset)
train_size = int(0.70 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(
    dataset, [train_size, val_size, test_size], generator=generator
)

NUM_WORKERS = min(2, os.cpu_count() or 1)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=NUM_WORKERS)
print(f'Train / Val / Test: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

## Model

In [ ]:
class YOLOBackbone(nn.Module):
    def __init__(self, model_name='yolov5su.pt', cut_layer=10):
        super().__init__()
        yolo = YOLO(model_name)
        self.feature_extractor = yolo.model.model[:cut_layer]
        for p in self.feature_extractor.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.feature_extractor(x).clone()


class SimpleFPN(nn.Module):
    def __init__(self, in_channels=512, out_channels=128):
        super().__init__()
        self.conv_out = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x): return self.conv_out(x)


class TaskEncoder(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.text_encoder = SentenceTransformer('all-MiniLM-L6-v2')
        for p in self.text_encoder.parameters():
            p.requires_grad = False
        self.linear = nn.Linear(384, output_dim)

    def forward(self, task_descriptions):
        dev = next(self.linear.parameters()).device
        emb = self.text_encoder.encode(task_descriptions, convert_to_tensor=True)
        emb = emb.detach().clone().to(dev)
        return F.relu(self.linear(emb))


class TransformerFusion(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=1, task_embed_dim=64):
        super().__init__()
        self.query_proj = nn.Linear(task_embed_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=False)
        self.transformer_encoder = nn.TransformerEncoder(
            enc_layer, num_layers=num_layers, enable_nested_tensor=False)

    def forward(self, vision_feats, task_embed):
        B, C, H, W = vision_feats.shape
        v_seq = rearrange(vision_feats, 'b c h w -> (h w) b c')
        t_q   = rearrange(self.query_proj(task_embed), 'b d -> 1 b d')
        enc   = self.transformer_encoder(torch.cat([t_q, v_seq], 0))
        return rearrange(enc[1:], '(h w) b c -> b c h w', h=H, w=W)


class SaliencyDecoder(nn.Module):
    def __init__(self, in_channels=128):
        super().__init__()
        self.conv1   = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.deconv1 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(32,  1, 4, stride=2, padding=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.deconv1(x))
        return torch.sigmoid(self.deconv2(x))


class YOLOTaskSaliencyModel(nn.Module):
    def __init__(self, task_embed_dim=64, vision_dim=128, nhead=4, num_layers=1):
        super().__init__()
        self.backbone           = YOLOBackbone()
        self.fpn                = SimpleFPN(512, vision_dim)
        self.task_encoder       = TaskEncoder(task_embed_dim)
        self.transformer_fusion = TransformerFusion(vision_dim, nhead, num_layers, task_embed_dim)
        self.saliency_decoder   = SaliencyDecoder(vision_dim)

    def forward(self, images, task_descriptions):
        feat  = self.fpn(self.backbone(images))
        t_emb = self.task_encoder(task_descriptions)
        fused = self.transformer_fusion(feat, t_emb)
        return self.saliency_decoder(fused)

## Loss

In [ ]:
class SaliencyLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, pred, gt):
        if pred.dim() == 4: pred = pred[:, 0]
        if gt.dim()   == 4: gt   = gt[:, 0]
        B   = pred.shape[0]
        p   = pred.reshape(B, -1)
        g   = gt.reshape(B, -1)
        eps = 1e-8

        p_n = (p + eps) / (p + eps).sum(1, keepdim=True)
        g_n = (g + eps) / (g + eps).sum(1, keepdim=True)
        kl  = (g_n * (g_n / p_n).log()).sum(1).mean()

        p_c = p - p.mean(1, keepdim=True)
        g_c = g - g.mean(1, keepdim=True)
        cc  = ((p_c * g_c).sum(1) /
               (torch.sqrt((p_c**2).sum(1) * (g_c**2).sum(1)) + eps)).mean()

        return self.alpha * kl + self.beta * (1.0 - cc)

## Training

In [ ]:
model      = YOLOTaskSaliencyModel().to(device)
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion  = SaliencyLoss(alpha=1.0, beta=1.0)
num_epochs = 50

In [ ]:
EPS    = 1e-8
_trapz = getattr(np, 'trapezoid', np.trapz)

def cc_metric(pred, gt):
    B = pred.shape[0]
    p = pred.view(B, -1) - pred.view(B, -1).mean(1, keepdim=True)
    g = gt.view(B, -1)   - gt.view(B, -1).mean(1, keepdim=True)
    return ((p * g).sum(1) / (torch.sqrt((p**2).sum(1) * (g**2).sum(1) + EPS))).mean().item()

def kl_metric(pred, gt):
    B = pred.shape[0]
    p = pred.view(B, -1).clamp(min=EPS); p = p / (p.sum(1, keepdim=True) + EPS)
    g = gt.view(B, -1).clamp(min=EPS);   g = g / (g.sum(1, keepdim=True) + EPS)
    return (g * (g.log() - p.log())).sum(1).mean().item()

def sim_metric(pred, gt):
    B = pred.shape[0]
    p = pred.view(B, -1).clamp(min=0); p = p / (p.sum(1, keepdim=True) + EPS)
    g = gt.view(B, -1).clamp(min=0);   g = g / (g.sum(1, keepdim=True) + EPS)
    return torch.min(p, g).sum(1).mean().item()

def nss_metric(pred, fix):
    B = pred.shape[0]
    s = pred.view(B, -1); f = fix.view(B, -1)
    s = (s - s.mean(1, keepdim=True)) / (s.std(1, keepdim=True) + EPS)
    return ((s * f).sum(1) / f.sum(1).clamp(min=1)).mean().item()

def auc_borji_metric(pred, fix, n_splits=100, step=0.1):
    pred = pred.detach().cpu().numpy()
    fix  = fix.detach().cpu().numpy()
    aucs = []
    for b in range(pred.shape[0]):
        s_map = pred[b, 0]
        f_map = fix[b, 0].astype(bool)
        if f_map.sum() == 0: continue
        S     = (s_map - s_map.min()) / (s_map.max() - s_map.min() + EPS)
        S_fix = S[f_map]
        neg_idx = np.where(~f_map.flatten())[0]
        if len(neg_idx) == 0: continue
        for _ in range(n_splits):
            S_rand = S.flatten()[
                np.random.choice(neg_idx, S_fix.size, replace=(len(neg_idx) < S_fix.size))
            ]
            th = np.arange(0, 1 + step, step)
            tp = np.array([(S_fix  >= t).mean() for t in th])
            fp = np.array([(S_rand >= t).mean() for t in th])
            aucs.append(-_trapz(tp, fp))
    return float(np.mean(aucs)) if aucs else float('nan')

def _append_if_finite(bucket, val):
    if np.isfinite(val): bucket.append(float(val))

def evaluate_model(m, loader, dev):
    m.to(dev).eval()
    scores = {k: [] for k in ['CC', 'KL', 'SIM', 'NSS', 'AUC-Borji']}
    with torch.no_grad():
        for batch in loader:
            imgs  = batch['stimuli'].to(dev)
            gts   = batch['fdm'].to(dev)
            descs = batch['task_description']
            preds = m(imgs, descs)
            preds = F.interpolate(preds, gts.shape[-2:], mode='bilinear', align_corners=False)
            fix   = (gts > 0.5).float()
            _append_if_finite(scores['CC'],        cc_metric(preds, gts))
            _append_if_finite(scores['KL'],        kl_metric(preds, gts))
            _append_if_finite(scores['SIM'],       sim_metric(preds, gts))
            _append_if_finite(scores['NSS'],       nss_metric(preds, fix))
            _append_if_finite(scores['AUC-Borji'], auc_borji_metric(preds, fix))
    return {k: float(np.mean(v)) if v else float('nan') for k, v in scores.items()}

In [ ]:
epoch_losses = []
val_metrics  = {k: [] for k in ['CC', 'KL', 'SIM', 'NSS', 'AUC-Borji']}

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        imgs  = batch['stimuli'].to(device)
        gts   = batch['fdm'].to(device)
        descs = batch['task_description']
        optimizer.zero_grad()
        preds = model(imgs, descs)
        preds = F.interpolate(preds, gts.shape[-2:], mode='bilinear', align_corners=False)
        loss  = criterion(preds, gts)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_losses.append(running_loss / len(train_loader))

    metrics = evaluate_model(model, val_loader, device)
    for k, v in metrics.items():
        val_metrics[k].append(v)

    print(f'Epoch {epoch:3d}/{num_epochs} | Loss: {epoch_losses[-1]:.4f} | '
          f'CC: {metrics["CC"]:.4f} | NSS: {metrics["NSS"]:.4f}')

In [ ]:
x = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(x, epoch_losses, marker='o')
axes[0].set(title='Training Loss', xlabel='Epoch', ylabel='Loss')
axes[0].grid(True, alpha=0.4)

for metric in val_metrics:
    axes[1].plot(x, val_metrics[metric], marker='o', label=metric)
axes[1].set(title='Validation Metrics', xlabel='Epoch', ylabel='Value')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## Save

In [ ]:
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
torch.save(model.state_dict(), CKPT_PATH)
print(f'Saved: {CKPT_PATH}')

## Evaluation

In [ ]:
def safe_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)

model.load_state_dict(safe_torch_load(CKPT_PATH, map_location=device))
test_metrics = evaluate_model(model, test_loader, device)

print(f'{"Metric":<12} {"Score":>8}')
print('-' * 22)
for k, v in test_metrics.items():
    print(f'{k:<12} {v:>8.4f}')

## Visualization

In [ ]:
def visualize_test_sample(mdl, ds, full_ds, n_tasks=4, alpha=0.55, cmap='jet'):
    dev = next(mdl.parameters()).device
    for _ in range(100):
        rand_idx = random.randint(0, len(ds) - 1)
        orig_idx = ds.indices[rand_idx]
        _, fdm_path, _ = full_ds.samples[orig_idx]
        base = os.path.splitext(os.path.basename(fdm_path))[0]
        matched = [i for i, (_, fp, _) in enumerate(full_ds.samples)
                   if os.path.splitext(os.path.basename(fp))[0] == base]
        if len(matched) == n_tasks:
            break
    else:
        print('Could not find a sample with all 4 tasks.')
        return

    samples = [full_ds[i] for i in matched]
    imgs_t  = torch.stack([s['stimuli'] for s in samples]).to(dev)
    fdm_t   = torch.stack([s['fdm'] for s in samples])
    descs   = [s['task_description'] for s in samples]

    with torch.no_grad():
        preds = mdl(imgs_t, descs)
        preds = F.interpolate(preds, fdm_t.shape[-2:], mode='bilinear', align_corners=False)

    img_np  = np.clip(imgs_t.cpu().permute(0, 2, 3, 1).numpy(), 0, 1)
    pred_np = preds.cpu().numpy()[:, 0]
    gt_np   = fdm_t.numpy()[:, 0]
    cm      = plt.get_cmap(cmap)

    def norm(x):
        mn = x.min(axis=(1, 2), keepdims=True)
        mx = x.max(axis=(1, 2), keepdims=True)
        return (x - mn) / (mx - mn + 1e-8)

    fig, axes = plt.subplots(2, n_tasks, figsize=(4 * n_tasks, 7))
    for i in range(n_tasks):
        pred_overlay = (1 - alpha) * img_np[i] + alpha * cm(norm(pred_np)[i])[..., :3]
        gt_overlay   = (1 - alpha) * img_np[i] + alpha * cm(norm(gt_np)[i])[..., :3]
        axes[0, i].imshow(np.clip(pred_overlay, 0, 1)); axes[0, i].axis('off')
        axes[0, i].set_title(descs[i], fontsize=9)
        axes[1, i].imshow(np.clip(gt_overlay, 0, 1));  axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Prediction', fontsize=10)
    axes[1, 0].set_ylabel('Ground Truth', fontsize=10)
    fig.suptitle(f'Test sample: {base}', fontsize=11)
    plt.tight_layout()
    plt.show()

visualize_test_sample(model, test_ds, dataset)